# V1DD stimulus-response metrics

Per-ROI tuning and responsiveness for every two-photon session in the V1DD functional
asset, computed from the NWB files and written as a derived data asset.

Six stimulus families, each answering a different question about a cell:

| family | what it measures |
|---|---|
| **drifting gratings, full field** | orientation and direction tuning across the whole screen |
| **drifting gratings, windowed** | the same, through a small aperture over the receptive field |
| **surround suppression** | how much the surround costs — the windowed response against the full-field one at the same condition |
| **natural images** | selectivity across 118 scenes, and how sparse that selectivity is |
| **natural images 12** | the same over a 12-image subset with far more repeats, so the responses are better estimated |
| **natural movie** | reliability across repeats of a continuous clip |
| **receptive fields** | which part of the screen a cell responds to, from locally sparse noise |

The output is one table per family plus a wide table joining them, keyed so it merges
directly against the EM coregistration.

**What this notebook does not do.** It computes and writes; it does not check itself.
The unit tests, the agreement against the historical `data_frames` tables, and the
seed-to-seed noise floor all live in `code/validation/`, and the reasoning behind the
harder choices is recorded there. Keeping the two apart means this notebook can be read
as a description of the analysis rather than an argument about it.

In [ ]:
import json
import os
import sys
import time
import traceback
from os.path import join as pjoin

import numpy as np
import pandas as pd
from IPython.display import display

for _candidate in [pjoin("..", "utils"), pjoin("code", "utils"), "utils"]:
    if os.path.isdir(_candidate):
        sys.path.append(os.path.abspath(_candidate))
        break
else:
    raise FileNotFoundError(f"could not locate 'utils'; cwd={os.getcwd()}")

import stimulus_metrics as sm
import trial_responses as tr
import v1dd_nwb as vn
from paths import resolve_data_root, resolve_dataset_dir
from provenance import git_sha, jsonable, latest_run, run_dir, run_stamp

pd.set_option("display.max_columns", None)
print(f"numpy {np.__version__} | pandas {pd.__version__}")

## Inputs and outputs

The mounted NWB asset is the only thing configured by hand. Everything else — which mouse,
which sessions, how many planes, the imaging depth of each — is read from the data, so
pointing this at a different animal's asset is the only change needed to run it there.

The seed matters. Three of the metrics (`z_score`, `is_responsive`,
`frac_responsive_trials`) compare each ROI against a bootstrapped null drawn from its own
spontaneous activity, so they are stochastic. Seeding explicitly and recording the seed is
what makes a rerun reproduce the asset rather than merely resemble it.

In [ ]:
functional_asset = "409828_V1DD_Filtered"   # the mounted NWB asset
SEED = 0                                    # bootstrap seed; recorded in the provenance

# None processes every session. A list of (column, volume) restricts it -- useful for a
# short run before committing to the full asset, and for regenerating one session after a
# fix. The subset is recorded in the provenance, so a partial asset cannot be mistaken
# for a complete one.
SESSION_FILTER = None       # e.g. [(1, "3"), (1, "5")] for the coregistered pair
# SESSION_FILTER = [(1, "3"), (1, "5")] 

data_root = resolve_data_root(functional_asset)
functional_dir = resolve_dataset_dir(functional_asset, root=data_root)

# "scratch" is ephemeral; "results" is what CodeOcean captures as a data asset.
# The capsule entry point sets SWDB_OUTPUT_TARGET=results for a reproducible run, so
# opening this notebook interactively cannot accidentally write something that looks
# like a captured asset.
output_target = os.environ.get("SWDB_OUTPUT_TARGET", "scratch")
save_root = f"/{output_target}"

print(f"functional_dir : {functional_dir}")
print(f"save_root      : {save_root}")

## Sessions

The asset stores most sessions as NWB-Zarr directories and a couple as plain HDF5 files.
A glob for `*.nwb.zarr` silently drops the latter — the symptom is a short session list
rather than an error — so discovery goes through `find_sessions`, which returns one path
per session across both, and `open_session` dispatches on the suffix. Everything
downstream sees the same `NWBFile` either way.

Each session covers one *column* (a cortical location) and one *volume* (a depth block of
six imaging planes).

In [ ]:
%%time
session_paths = vn.find_sessions(functional_dir)
inventory = pd.DataFrame([vn.peek_session(p) for p in session_paths])
inventory["format"] = [vn.nwb_format(p) for p in session_paths]

bad = inventory[inventory["error"].notna()]
if len(bad):
    print(f"!! {len(bad)} session(s) could not be opened and will be skipped:")
    display(bad[["name", "format", "error"]])
sessions = inventory[inventory["error"].isna()].reset_index(drop=True)
if SESSION_FILTER is not None:
    want = {(int(c), str(v)) for c, v in SESSION_FILTER}
    sessions = sessions[[(int(c), str(v)) in want
                         for c, v in zip(sessions["column"], sessions["volume"])]
                        ].reset_index(drop=True)
    print(f"SESSION_FILTER active: {len(sessions)} of {len(inventory)} session(s)")
    if len(sessions) != len(want):
        print(f"!! asked for {len(want)} session(s), matched {len(sessions)}")

# The mouse comes from the file, not from a constant. Every session must agree: this asset
# is per-animal, so a disagreement means something is wrong rather than that we are
# processing two mice.
with vn.session(sessions["path"].iloc[0]) as _nwb:
    mouse_id, mouse_label = vn.session_mouse(_nwb, sessions["path"].iloc[0])

asset_name = f"{mouse_id}_V1DD_stimulus_metrics"
RUN_STAMP = run_stamp()
previous_run = latest_run(save_root, asset_name)
save_dir = run_dir(save_root, asset_name, RUN_STAMP)
os.makedirs(save_dir, exist_ok=True)

print(f"{len(sessions)} session(s), {int(sessions['n_planes'].sum())} planes, "
      f"mouse {mouse_label}")
print(f"  formats  {inventory['format'].value_counts().to_dict()}")
print(f"  columns  {sorted(sessions['column'].dropna().astype(int).unique())}")
print(f"  volumes  {sorted(sessions['volume'].dropna().astype(str).unique())}")
print(f"save_dir     : {save_dir}")
print(f"previous run : {previous_run or 'none'}")
display(sessions[["name", "format", "column", "volume", "n_planes"]])

## What counts as a response

Every metric below reduces a trial to one number: the mean of the trace over a window
starting at stimulus onset. Three choices define that reduction, and they are the same
everywhere unless noted.

**Deconvolved events, not ΔF/F.** Events are a sparse estimate of spiking, already
deconvolved from the calcium transient, so a trial mean is close to a rate rather than to
a slow decaying fluorescence. The one exception is receptive fields, which use ΔF/F with
the preceding second subtracted — that method needs a signal that varies smoothly enough
for a baseline to mean something.

**No baseline subtraction.** Events sit near zero between transients, so a baseline term
would mostly add noise. Receptive fields, again, are the exception.

**Windows are counted in imaging samples where the response is brief.** Natural movie
(3 samples), locally sparse noise (4) and natural images (2) all use a fixed sample count;
drifting gratings use 2.0 s, which spans about twelve samples. The distinction matters
more than it looks: at this frame rate a *time* window catches a different number of
samples depending on where the onset falls relative to the sampling clock, and that
rescales each trial slightly differently. For a two-sample response that is the difference
between a clean measurement and a noisy one, so the short windows are specified in the
units the intent actually lives in. Sampling period varies between sessions by about 3 %,
which is enough to matter.

**Responsiveness is measured against the cell's own spontaneous activity.** Each session
contains one block of grey screen. Sampling windows from that block, with the same width
as the stimulus window, gives a per-ROI null distribution; a trial counts as a response
when it exceeds the 95th percentile of that null. This is per-ROI rather than pooled,
because baseline event rates vary enormously between cells.

In [ ]:
CONFIG = sm.MetricConfig()          # documented in code/utils/stimulus_metrics.py

_c = CONFIG
print("response windows")
print(f"  drifting gratings   {_c.dg_response_seconds} s")
print(f"  natural images      {_c.ni_response_frames} imaging samples")
print(f"  natural movie       {_c.nm_response_frames} imaging samples")
print(f"  locally sparse noise {_c.lsn_response_frames} imaging samples")
print("bootstrap")
print(f"  drifting gratings   {_c.dg_n_boot} draws")
print(f"  other families      {_c.other_n_boot} draws")
print(f"  significance        p < {_c.sig_p_thresh}")
print("traces")
for fam, kind in sorted(_c.trace_type.items()):
    print(f"  {fam:<28} {kind}")

## The families in more detail

### Drifting gratings, and surround suppression

Twelve directions crossed with two spatial frequencies, eight trials each, shown full
field and again through a small window. Interleaved blank (grey) sweeps give a
within-stimulus baseline.

From the trial means at each condition come the standard selectivity measures — `osi` and
`dsi` compare the preferred direction against the orthogonal and opposite ones, `gosi` is
the vector-sum version over all directions, `lifetime_sparseness` describes how much of
the response is concentrated in few conditions. A von Mises curve with two opposed peaks
is fitted per spatial frequency, giving a preferred direction that is not limited to the
twelve sampled ones.

**Surround suppression compares the two stimuli at the same condition.** The reference is
always the *windowed* stimulus's preferred direction and spatial frequency, and the
full-field response is read at that same condition rather than at its own preferred one —
the question is what the surround does to a given drive, not which stimulus a cell prefers.
The index is `(W − F) / (W + F)`: positive means the surround suppresses.

Eight variants are reported, differing in what is averaged and whether the animal is
moving. Running and stationary split at 1 cm/s with strict inequalities on both sides, so
a trial at exactly 1 cm/s belongs to neither. Two of the variants (`ssi_running`,
`ssi_stationary`) additionally require at least three qualifying trials in *both* stimuli
and are NaN otherwise; the `*_avg_at_pref_sf` variants have no such minimum, which is why
they are populated far more often.

### Natural images

118 scenes, eight repeats each; and a 12-image subset with forty repeats, which trades
breadth for much better-estimated per-image responses. `pref_img` is the best image,
`pref_response` its mean, `lifetime_sparseness` how peaked the response is across images,
and `z_score` how far the preferred response sits above the spontaneous null.

### Natural movie

A continuous clip, 3,600 frames, repeated nine times. Each frame is treated as a
condition, so `pref_img` is a frame index rather than an image id.

`frac_responsive_trials` means something different here than elsewhere: it is the fraction
of repeats with any response above zero — no null, no threshold, no randomness. It is
therefore the one fully deterministic responsiveness measure in the asset.

One consequence of treating frames as conditions is worth knowing before interpreting
`pref_img`: the response window extends forward over several frames, and consecutive
frames are 1/30 s apart, so activity driven by one frame falls inside the windows of
several neighbouring ones. The preferred frame is best read as "somewhere around here in
the clip", not as the exact frame that drove the cell.

### Receptive fields

Locally sparse noise on an 8 × 14 grid at 9.3°, with a few pixels bright and a few dark on
each presentation. For every pixel, the fraction of its bright presentations that produced
a response above the ROI's spontaneous 95th percentile becomes that pixel's value in the ON
map; dark presentations give the OFF map. Fractions below 0.25 are zeroed, so "has a
receptive field" reduces to "at least one pixel survived", and the centre is the
**unweighted** centroid of the surviving pixels.

Despite the name sometimes given to this method, there is no regression involved. The
design matrix records which pixels were bright and which dark on each sweep, and is then
used purely as a counting indicator.

The ON/OFF pixel values are read from the template rather than assumed. This asset encodes
them as −1 / 0 / +1; hard-coding a different convention does not fail loudly, it silently
empties the ON map and turns the OFF map into a map of the background.

## Two departures from the historical tables

The `data_frames/*_M409828.csv` tables distributed previously were produced by an earlier
pipeline. This one reproduces them where they were right and departs where they were not.
Both departures are single flags in `MetricConfig`, so the old behaviour stays reachable —
`sm.REFERENCE_CONFIG` restores it — and `code/validation/` checks that it still does.

**Receptive-field centres are on the true degree scale.** The earlier mapping divided the
centre-to-centre range of the pixel grid by the number of pixels rather than the number of
gaps between them, compressing the map by `(n−1)/n`: 12.5 % in altitude, 7.1 % in azimuth.
The historical tables therefore span ±28.481° and ±56.132° where the screen actually spans
±32.55° and ±60.45°. Anyone plotting retinotopy from those numbers plots it wrong, so the
default here is the true mapping. The two differ by exactly `n/(n−1)`.

**A cell with no response has no preferred condition.** `preferred_dir` and `preferred_sf`
were taken from an argmax over responses with missing values filled to −1, so an ROI with
no finite response at any condition reported direction 0 — a real-looking value with
nothing behind it. Those ROIs are now NaN. Surround suppression keys off the preferred
condition, so the fabricated value propagated.

A third change is not a correction but a generalisation: the natural-images window is
specified as two imaging samples rather than 0.33 s. The two agree on the sessions the
value was originally derived from, and only there — sampling period varies enough across
the asset that the same duration catches one or three samples elsewhere.

## Processing

One pass over the asset. Each plane is loaded once, with both trace types, and all six
families are computed from it before it is released — the alternative, a pass per family,
would reopen every file six times for no benefit.

Memory stays flat: one plane's traces are live at a time, so peak usage is set by the
largest single plane rather than by the number of sessions.

A session that fails partway is recorded and skipped rather than aborting the run. With
25 sessions and a couple of hours of work, losing everything to one bad file would be a
poor trade.

In [ ]:
%%time
FAMILIES = ["roi_summary", "drifting_gratings_full", "drifting_gratings_windowed",
            "surround_supression_index", "natural_images", "natural_images_12",
            "natural_movie", "rf_metrics"]
parts = {f: [] for f in FAMILIES}
rf_maps, lsn_grid = [], None   # RF map accumulator and LSN coordinate grid
# Grating tuning curves. `drifting_gratings_metrics` already builds the full
# (n_rois, n_dir, n_sf, n_trials) response array and returns it in DGResult; without this
# it is consumed by surround suppression and dropped, leaving only the six reduced
# columns. Keeping it is retention, not computation.
TUNING_PARTS = ("trials", "blank", "params", "running", "plane_key")
tuning = {k: {p: [] for p in TUNING_PARTS} for k in ("dgw", "dgf")}
tuning_axes = None             # (directions, spatial_frequencies), set on the first plane
failures, plane_log = [], []
t_start = time.time()

for n_session, (_, srow) in enumerate(sessions.iterrows(), 1):
    t_sess = time.time()
    try:
        nwb, io = vn.open_session(srow["path"])
        try:
            stim = vn.load_stimulus_table(nwb)
            spont = vn.spontaneous_block(nwb)
            running = vn.load_running_speed(nwb)
            dg_trials = {t: vn.stimulus_trials(stim, f"drifting_gratings_{t}",
                                               vn.DG_PARAM_COLUMNS)
                         for t in ("full", "windowed")}
            ni_trials = {f: vn.stimulus_trials(stim, f)[0]
                         for f in ("natural_images", "natural_images_12")}
            nm_trials, _ = vn.stimulus_trials(stim, "natural_movie")
            lsn_trials, _ = vn.stimulus_trials(stim, "locally_sparse_noise")
            lsn = vn.load_lsn_template(nwb)
            if lsn_grid is None:
                lsn_grid = {"altitudes": np.asarray(lsn["altitudes"], dtype=np.float64),
                            "azimuths": np.asarray(lsn["azimuths"], dtype=np.float64)}

            for plane_key in vn.list_planes(nwb):
                plane = vn.load_plane(nwb, plane_key, trace_types=("events", "dff"))
                rng = lambda: np.random.default_rng(SEED)      # noqa: E731

                # Receptive fields FIRST, because surround suppression now reports how
                # much of each field the grating aperture actually covered. `rng` is a
                # factory returning a freshly seeded generator per family, so reordering
                # the calls changes no number -- verify with diff_runs.py, not by trust.
                rf_df, rf_map = sm.receptive_field_metrics(
                    plane, lsn_trials, spont, lsn, config=CONFIG, rng=rng())
                parts["rf_metrics"].append(rf_df)
                rf_maps.append(rf_map)

                dg = {}
                # windowed first -- self-selects its preferred SF for the tuning fit (~2x speedup)
                t, blank = dg_trials["windowed"]
                dg["windowed"] = sm.drifting_gratings_metrics(
                    plane, t, blank, spont, running, dg_type="windowed",
                    config=CONFIG, rng=rng())
                parts["drifting_gratings_windowed"].append(dg["windowed"].metrics)
                # full field -- only fits the SF that SSI reads (windowed's preferred per ROI)
                t, blank = dg_trials["full"]
                dg["full"] = sm.drifting_gratings_metrics(
                    plane, t, blank, spont, running, dg_type="full",
                    fit_sf_index=dg["windowed"].pref_cond_index[:, 1],
                    config=CONFIG, rng=rng())
                parts["drifting_gratings_full"].append(dg["full"].metrics)
                # RF-vs-aperture geometry: reported, never used to filter. See the
                # window_containment docstring for why both a distance and an overlap.
                containment = sm.window_containment(rf_df, rf_map, lsn,
                                                    dg["windowed"].center, config=CONFIG)
                parts["surround_supression_index"].append(
                    sm.surround_suppression_metrics(dg["windowed"], dg["full"], plane,
                                                    config=CONFIG,
                                                    containment=containment))
                # Locomotion spans stimuli -- both grating types plus the
                # spontaneous block -- so it is its own family rather than columns bolted
                # onto one of them. Needs both DGResults, so it runs before `del dg`.
                parts["roi_summary"].append(
                    sm.roi_summary_metrics(plane, dg["windowed"], dg["full"], spont,
                                          running, config=CONFIG))
                for _key, _res in (("dgw", dg["windowed"]), ("dgf", dg["full"])):
                    _acc = tuning[_key]
                    _acc["trials"].append(_res.trial_responses.astype(np.float32))
                    _acc["blank"].append(_res.blank_responses.astype(np.float32))
                    _acc["params"].append(_res.tuning_params.astype(np.float32))
                    _acc["running"].append(_res.trial_running_speeds.astype(np.float32))
                    # Running speeds have no ROI axis, so they key on the plane. Taken as
                    # roi_key's prefix rather than rebuilt from parts, so the two cannot
                    # drift: roi_key is M{mouse}_{column}_{volume}_{plane}_{roi}.
                    _acc["plane_key"].append(
                        _res.metrics["roi_key"].iloc[0].rsplit("_", 1)[0]
                        if len(_res.metrics) else "")
                if tuning_axes is None:
                    tuning_axes = (dg["windowed"].dir_list, dg["windowed"].sf_list)
                del dg, containment, rf_df

                for fam in ("natural_images", "natural_images_12"):
                    parts[fam].append(sm.natural_images_metrics(
                        plane, ni_trials[fam], spont, ns_type=fam, config=CONFIG,
                        rng=rng()))
                parts["natural_movie"].append(sm.natural_movie_metrics(
                    plane, nm_trials, spont, config=CONFIG, rng=rng()))

                plane_log.append({"session": srow["name"], "column": plane.column,
                                  "volume": plane.volume, "plane": plane.plane,
                                  "n_rois": plane.n_rois, "depth_um": plane.depth_um,
                                  "dt": round(plane.dt, 6)})
                del plane
        finally:
            io.close()
        print(f"  [{n_session:>2}/{len(sessions)}] col{srow['column']} vol{srow['volume']} "
              f"{srow['name'][:34]:<34} {time.time() - t_sess:>6.1f}s")
    except Exception as exc:                                   # noqa: BLE001
        failures.append({"name": srow["name"], "error": f"{type(exc).__name__}: {exc}",
                         "traceback": traceback.format_exc(limit=3)})
        print(f"  [{n_session:>2}/{len(sessions)}] {srow['name'][:34]:<34} FAILED: {exc}")

wall_seconds = time.time() - t_start
tables = {f: pd.concat(v, ignore_index=True) for f, v in parts.items() if v}
planes = pd.DataFrame(plane_log)
print(f"\n{len(planes)} planes, {int(planes['n_rois'].sum())} ROIs, "
      f"{wall_seconds / 60:.1f} min")
if failures:
    print(f"!! {len(failures)} session(s) failed: {[f['name'] for f in failures]}")

In [ ]:
# Every family must cover exactly the same ROIs. They are built from the same planes in
# the same order, so anything else means a family silently dropped or duplicated rows.
KEYS = ["column", "volume", "plane", "roi"]
ref_keys = set(map(tuple, tables["natural_movie"][KEYS].astype({"volume": str}).to_numpy().tolist()))
for fam, t in tables.items():
    got = set(map(tuple, t[KEYS].astype({"volume": str}).to_numpy().tolist()))
    if got != ref_keys:
        raise AssertionError(f"{fam}: ROI set differs -- {len(got - ref_keys)} extra, "
                             f"{len(ref_keys - got)} missing")
    if len(t) != len(ref_keys):
        raise AssertionError(f"{fam}: {len(t)} rows for {len(ref_keys)} distinct ROIs")
print(f"all seven families cover the same {len(ref_keys)} ROIs")

summary = planes.groupby(["column", "volume"]).agg(
    planes=("plane", "count"), rois=("n_rois", "sum"),
    depth_min=("depth_um", "min"), depth_max=("depth_um", "max"),
    dt=("dt", "median")).reset_index()
display(summary)

## Outputs

Seven per-family tables, one row per ROI, plus a wide table joining them.

The wide table is the one to merge against the coregistration: five of the seven families
carry a column called `lifetime_sparseness`, so a naive concatenation either collides or
silently suffixes. Each family therefore gets a prefix, except the two whose published
column names already carry theirs — the surround-suppression columns all begin `ssi`, and
the receptive-field columns all contain `rf`.

Identity columns are shared: `roi_unique_id` keeps the historical format, which omits the
column and so is **not unique across the asset**; `roi_key` includes it and is. Join on
`(column, volume, plane, roi)`.

`pika_roi_confidence` is carried through from the segmentation. ROIs at or below 0.5 are
treated as unreliable, and the pipeline suppresses the columns that depend on choosing a
preferred condition for them — `preferred_dir`, `preferred_sf`, every `ssi*` and every
receptive-field column — while leaving the rest populated. Emitting the confidence makes
that visible; without it those ROIs are neither dropped nor labelled, and they enter any
population average unnoticed.

In [ ]:
written = {}
for fam in FAMILIES:
    out = sm.to_output_schema(tables[fam], fam)
    path = pjoin(save_dir, f"{fam}_{mouse_label}.csv")
    out.to_csv(path, index=False)
    written[fam] = {"file": os.path.basename(path), "rows": int(len(out)),
                    "columns": list(out.columns)}
    print(f"  {os.path.basename(path):<52} {len(out):>6} rows")

In [ ]:
ID_COLS = ["roi_unique_id", "roi_key", "mouse", "column", "volume", "plane", "roi",
           "depth_um", "pika_roi_confidence"]
PREFIX = {"drifting_gratings_full": "dgf_", "drifting_gratings_windowed": "dgw_",
          "surround_supression_index": "", "roi_summary": "", "natural_images": "ni_",
          "natural_images_12": "ni12_", "natural_movie": "nm_", "rf_metrics": ""}

def _keyed(df):
    out = df.copy()
    out["volume"] = out["volume"].astype(str)
    for k in ("column", "plane", "roi"):
        out[k] = out[k].astype("int64")
    return out

wide = _keyed(tables["natural_movie"][ID_COLS])
manifest = {}
for fam in FAMILIES:
    pub = _keyed(sm.to_output_schema(tables[fam], fam))
    metric_cols = [c for c in pub.columns if c not in ID_COLS]
    prefix = PREFIX[fam]
    part = pub[KEYS + metric_cols].rename(columns={c: prefix + c for c in metric_cols})
    # one_to_one is the assertion that matters: it fails loudly rather than fanning the
    # table out if any family has a duplicate key.
    wide = wide.merge(part, on=KEYS, how="left", validate="one_to_one")
    manifest[fam] = {"prefix": prefix, "columns": [prefix + c for c in metric_cols]}

if not wide.columns.is_unique:
    raise AssertionError(f"duplicate columns: {wide.columns[wide.columns.duplicated()].tolist()}")

wide_path = pjoin(save_dir, f"stimulus_metrics_{mouse_label}.feather")
wide.to_feather(wide_path)
print(f"wrote {os.path.basename(wide_path)}: {len(wide)} ROIs x {len(wide.columns)} columns "
      f"({len(ID_COLS)} identity + {len(wide.columns) - len(ID_COLS)} metrics)")
display(wide.head(3))

In [ ]:
# Receptive-field maps: pre-threshold continuous fraction, shape (n_rois, 2, n_rows, n_cols).
# Each value is the fraction of that pixel's presentations that produced a response above
# the per-ROI bootstrapped spontaneous 95th percentile, before zeroing pixels below 0.25.
# Invalid ROIs (pika_roi_confidence <= 0.5) are all-zero; blank = excluded, not no RF.
#
# Three arrays travel with the maps or they are uninterpretable:
#   roi_key     -- unique per-ROI string, joins to the wide table on "roi_key"
#   altitudes   -- (n_rows,) degrees of visual angle, pixel row index -> elevation
#   azimuths    -- (n_cols,) degrees of visual angle, pixel col index -> azimuth
#   seed        -- bootstrap seed; significance per pixel is seed-dependent
# Array outputs travel beside the tables. Collected so the manifest can check they
# landed and the provenance can name them -- neither did before.
extra_outputs = []
if rf_maps and lsn_grid is not None:
    rf_map_arr = np.concatenate(rf_maps, axis=0).astype(np.float32)
    roi_key_arr = tables["rf_metrics"]["roi_key"].to_numpy()
    rf_map_path = pjoin(save_dir, f"rf_maps_{mouse_label}.npz")
    np.savez_compressed(rf_map_path, rf_maps=rf_map_arr, roi_key=roi_key_arr,
                        altitudes=lsn_grid["altitudes"], azimuths=lsn_grid["azimuths"],
                        seed=np.array(SEED))
    extra_outputs.append(os.path.basename(rf_map_path))
    print(f"wrote {os.path.basename(rf_map_path)}: {rf_map_arr.shape} ",
          f"({os.path.getsize(rf_map_path) / 1e6:.1f} MB)")
else:
    print("!! rf_maps is empty -- no .npz written")

In [ ]:
# Grating tuning curves: the full per-trial response to every (direction, spatial
# frequency), for both grating types. The published CSVs reduce this to six numbers
# (preferred_dir, preferred_sf, osi, dsi, gosi, pref_dir_mean); this is what those six
# were computed from, so any of them can be checked by eye or recomputed.
#
#   dg{w,f}_trials    (n_rois, n_dir, n_sf, n_trials)  response per trial, NaN-padded
#   dg{w,f}_blank     (n_rois, n_blank)                interleaved grey sweeps -- the
#                                                      baseline to draw under a curve
#   dg{w,f}_params    (n_rois, n_sf, 6)                von Mises fit, scipy parameter order
#   dg{w,f}_running   (n_planes, n_dir, n_sf, n_trials)  cm/s -- NO roi axis, keys on plane
#   roi_key           (n_rois,)    joins to the wide table
#   plane_key         (n_planes,)  roi_key minus its trailing _{roi}; indexes *_running
#   directions, spatial_frequencies   axis labels in degrees and cycles/degree
#
# Trial-level rather than trial means: means are one nanmean away, the reverse is not, and
# error bars, per-SF curves and any recomputation need the trials. Same reasoning that
# made the pre-threshold receptive-field map the right thing to keep.
#
# Read `dgw_params` knowing that only the SF surround suppression consumes is fitted --
# the other column is NaN by design (the fit-only-the-used-SF speedup), not by failure.
if all(tuning[k]["trials"] for k in ("dgw", "dgf")) and tuning_axes is not None:
    payload, shapes = {}, {}
    for key in ("dgw", "dgf"):
        acc = tuning[key]

        # Concatenation along the ROI axis is only meaningful if every plane agrees on the
        # trailing dimensions. n_dir is already enforced by a raise inside the metric
        # function; n_sf and n_trials were uniform across all 25 sessions in the pre-flight.
        # n_blank never was -- blank sweeps are intermingled and the verified 192-sweep
        # count is the non-blank total -- so a session with a different blank count is
        # something to be told about rather than to pad over.
        for part, axes in (("trials", (1, 2, 3)), ("blank", (1,)), ("params", (1, 2))):
            seen = {tuple(a.shape[i] for i in axes) for a in acc[part]}
            if len(seen) != 1:
                raise AssertionError(
                    f"{key}_{part}: planes disagree on shape {sorted(seen)} -- "
                    f"cannot concatenate along the ROI axis")
        run_shapes = {a.shape for a in acc["running"]}
        if len(run_shapes) != 1:
            raise AssertionError(f"{key}_running: planes disagree on shape {run_shapes}")

        payload[f"{key}_trials"] = np.concatenate(acc["trials"], axis=0)
        payload[f"{key}_blank"] = np.concatenate(acc["blank"], axis=0)
        payload[f"{key}_params"] = np.concatenate(acc["params"], axis=0)
        payload[f"{key}_running"] = np.stack(acc["running"], axis=0)
        shapes[key] = payload[f"{key}_trials"].shape

    roi_key_arr = tables["drifting_gratings_windowed"]["roi_key"].to_numpy()
    plane_key_arr = np.asarray(tuning["dgw"]["plane_key"])
    if payload["dgw_trials"].shape[0] != len(roi_key_arr):
        raise AssertionError(
            f"tuning curves cover {payload['dgw_trials'].shape[0]} ROIs but the windowed "
            f"grating table has {len(roi_key_arr)}")
    if shapes["dgw"] != shapes["dgf"]:
        raise AssertionError(f"windowed {shapes['dgw']} and full-field {shapes['dgf']} "
                             f"tuning arrays differ in shape")
    if payload["dgw_running"].shape[0] != len(plane_key_arr):
        raise AssertionError("running speeds and plane keys are out of step")
    # every ROI's key must sit under its plane's key, which is the join the file promises
    if not np.isin(np.array([k.rsplit("_", 1)[0] for k in roi_key_arr]),
                   plane_key_arr).all():
        raise AssertionError("roi_key prefixes do not all appear in plane_key")

    tuning_path = pjoin(save_dir, f"tuning_curves_{mouse_label}.npz")
    np.savez_compressed(
        tuning_path, roi_key=roi_key_arr, plane_key=plane_key_arr,
        directions=np.asarray(tuning_axes[0], dtype=np.float64),
        spatial_frequencies=np.asarray(tuning_axes[1], dtype=np.float64),
        trace_type=np.array(CONFIG.trace_type["drifting_gratings_windowed"]),
        **payload)
    extra_outputs.append(os.path.basename(tuning_path))
    print(f"wrote {os.path.basename(tuning_path)}: "
          f"trials {shapes['dgw']} x 2 grating types, "
          f"blank {payload['dgw_blank'].shape[1]} sweeps, "
          f"running {payload['dgw_running'].shape} "
          f"({os.path.getsize(tuning_path) / 1e6:.1f} MB)")
else:
    print("!! no tuning curves accumulated -- no .npz written")

In [ ]:
import dataclasses
import platform


def config_dict(cfg):
    """MetricConfig -> plain dict. `dataclasses.asdict` cannot copy the MappingProxyType."""
    d = {f.name: getattr(cfg, f.name) for f in dataclasses.fields(sm.MetricConfig)}
    d["trace_type"] = dict(d["trace_type"])
    return d


def _ver(name):
    try:
        return __import__(name).__version__
    except Exception:                                          # noqa: BLE001
        return None


defaults = config_dict(CONFIG)
reference = config_dict(sm.REFERENCE_CONFIG)

provenance = {
    "asset": asset_name, "run_stamp": RUN_STAMP,
    "generated_utc": pd.Timestamp.utcnow().isoformat(timespec="seconds"),
    "git_sha": git_sha(), "mouse": mouse_label, "seed": SEED,
    "input_asset": str(functional_dir),
    "n_sessions": int(len(sessions)), "n_planes": int(len(planes)),
    "session_filter": SESSION_FILTER,
    "complete_asset": bool(SESSION_FILTER is None and not failures),
    "n_rois": int(len(wide)), "wall_seconds": round(wall_seconds, 1),
    "sessions": [{"name": r["name"], "format": r["format"], "column": int(r["column"]),
                  "volume": str(r["volume"]), "n_planes": int(r["n_planes"])}
                 for _, r in sessions.iterrows()],
    "failed_sessions": failures,
    "config": defaults,
    # What would have to change to reproduce the historical tables instead. Recording the
    # delta rather than prose means the claim stays checkable.
    "differs_from_reference_config": {k: {"used": defaults[k], "historical": reference[k]}
                                      for k in defaults if defaults[k] != reference[k]},
    "outputs": {"per_family": written, "wide_table": os.path.basename(wide_path),
                "wide_columns": manifest, "arrays": sorted(extra_outputs)},
    "environment": {"python": platform.python_version(), "platform": platform.platform(),
                    "packages": {m: _ver(m) for m in ("numpy", "pandas", "scipy", "pynwb",
                                                      "hdmf", "hdmf_zarr", "h5py", "zarr",
                                                      "pyarrow")}},
}
prov_path = pjoin(save_dir, "stimulus_metrics_provenance.json")
with open(prov_path, "w", encoding="utf-8") as fh:
    json.dump(jsonable(provenance), fh, indent=2, sort_keys=True, allow_nan=False)
    fh.write("\n")
print(f"wrote {os.path.basename(prov_path)}")
print()
print("settings that differ from the historical pipeline:")
for k, v in provenance["differs_from_reference_config"].items():
    print(f"  {k:<22} used {v['used']!r}, historically {v['historical']!r}")

In [ ]:
print(save_dir)
for f in sorted(os.listdir(save_dir)):
    p = pjoin(save_dir, f)
    print(f"  {f:<52} {os.path.getsize(p) / 1024:>9.1f} KB")

expected = [w["file"] for w in written.values()] + [
    os.path.basename(wide_path), "stimulus_metrics_provenance.json"] + extra_outputs
missing = [f for f in expected if not os.path.isfile(pjoin(save_dir, f))]
print()
print("all expected outputs present" if not missing else f"!! missing: {missing}")
if failures:
    print(f"!! {len(failures)} session(s) failed -- the asset is incomplete")
if SESSION_FILTER is not None:
    print(f"!! SESSION_FILTER was set -- this run covers {len(sessions)} session(s), "
          f"not the whole asset (recorded as complete_asset=false in the provenance)")

## Using the result

Merge the wide table against the coregistration on `(column, volume, plane, roi)`:

```python
metrics = pd.read_feather(".../stimulus_metrics_M409828.feather")
merged = coreg_df.merge(metrics, on=["column", "volume", "plane", "roi"], how="inner")
```

`coreg_df` contains duplicate rows in some materializations, which inflate counts after a
merge — deduplicate on the join key first.

Only a subset of these sessions is EM-coregistered; the rest are valid functional
measurements with no synaptic partner to merge against. `depth_um` gives the imaging depth
of each plane, which `(column, volume, plane)` only encodes implicitly, and spans roughly
50–515 µm across the asset.

**Response windows are inherited, not optimised.** They were chosen for the slower calcium
signals an earlier pipeline worked with, and this one runs on deconvolved events, which are
considerably sparser. A window that suits one is not obviously right for the other. They
are a `MetricConfig` field, so revisiting them is a one-line change — and
`code/validation/` provides the machinery to say whether a change helped.